In [31]:
from games.leduc import Leduc
from agents.counterfactualregret_t import CounterFactualRegret
from agents.agent_random import RandomAgent
from agents.ismcts import ISMCTS
import numpy as np
from collections import defaultdict
from tqdm import tqdm

In [32]:
def mapActionName(action):
    return ['call', 'raise', 'fold', 'check'][action]

In [33]:
game = Leduc(render_mode="human")
game.reset(seed=4)

print("Agents:", game.agents)
print("Current agent:", game.agent_selection)

print("Observation player_0", game.observe(game.agents[0]))
print("Observation player_1", game.observe(game.agents[1]))

print("Legal actions:", [mapActionName(action) for action in game.available_actions()])

Agents: ['player_0', 'player_1']
Current agent: player_1
Observation player_0 K_#_2_1_1
Observation player_1 J_#_2_1_1
Legal actions: ['call', 'raise', 'fold']


In [25]:
agents_random = {
    agent: RandomAgent(game=game, agent=agent)
    for agent in game.agents
}
agents_random

{'player_0': <agents.agent_random.RandomAgent at 0x276a8d46d50>,
 'player_1': <agents.agent_random.RandomAgent at 0x276a9883a90>}

In [26]:
game.reset(seed=1)

while not game.game_over():
    agent = game.agent_selection
    legal_actions = game.available_actions()
    action = agents_random[agent].action()

    game.render()
    print(f"Agent {agent}")
    print(f"Observation: {game.observe(agent)}")
    print(f"Legal actions: {[mapActionName(action) for action in legal_actions]}")
    print(f"Action '{mapActionName(action)}'")
    print()

    game.step(action)

game.render()
for agent in game.agents:
    print(f"Reward {agent} = {game.reward(agent)}")

Agent player_0
Observation: K_#_1_2_0
Legal actions: ['call', 'raise', 'fold']
Action 'fold'

Reward player_0 = -0.5
Reward player_1 = 0.5


In [ ]:
game = Leduc(render_mode="")
game.reset(seed=1)

agents = [
    CounterFactualRegret(game=game, agent=game.agents[0]),
    #ISMCTS(game=game, agent=game.agents[1], simulations=200, rollouts=10, rollout_iterations=100)
    CounterFactualRegret(game=game, agent=game.agents[1])
]
agents

In [39]:
n_training_iterations = 100

for i, agent in enumerate(game.agents):
    print(f"Training agent {agent}")
    if hasattr(agents[i], 'train'):
        agents[i].train(n_training_iterations)
        print(f"Known information sets: {len(agents[i].node_dict)}")

Training agent player_0


100%|██████████| 100/100 [00:36<00:00,  2.77it/s]

Known information sets: 576
Training agent player_1


In [40]:
for i, agent in enumerate(game.agents):
    print(f"First 10 policies for {agent} (total information sets: {len(agents[i].node_dict)})")
    for obs, node in list(agents[i].node_dict.items())[:10]:
        policy = [(mapActionName(move), round(float(prob), 3)) for move, prob in enumerate(node.policy(node.game.available_actions()))]
        print(obs, policy)
    print()

First 10 policies for player_0 (total information sets: 576)
K_#_1_2_0 [('call', 0.209), ('raise', 0.778), ('fold', 0.013), ('check', 0.0)]
Q_#_2_2_0c [('call', 0.0), ('raise', 0.128), ('fold', 0.025), ('check', 0.847)]
K_#_2_4_0cr [('call', 0.434), ('raise', 0.553), ('fold', 0.013), ('check', 0.0)]
Q_K_4_4_0crc [('call', 0.0), ('raise', 0.53), ('fold', 0.029), ('check', 0.442)]
K_K_4_8_0crcr [('call', 0.082), ('raise', 0.886), ('fold', 0.032), ('check', 0.0)]
Q_K_12_8_0crcrr [('call', 0.971), ('raise', 0.0), ('fold', 0.029), ('check', 0.0)]
K_K_4_4_0crck [('call', 0.0), ('raise', 0.879), ('fold', 0.032), ('check', 0.089)]
Q_K_8_4_0crckr [('call', 0.375), ('raise', 0.596), ('fold', 0.029), ('check', 0.0)]
K_K_8_12_0crckrr [('call', 0.967), ('raise', 0.0), ('fold', 0.033), ('check', 0.0)]
Q_#_6_4_0crr [('call', 0.985), ('raise', 0.0), ('fold', 0.015), ('check', 0.0)]



AttributeError: 'ISMCTS' object has no attribute 'node_dict'

In [45]:
cum_rewards = defaultdict(float)
n_games = 100

for _ in tqdm(range(n_games)):
    game.reset()
    while not game.game_over():
        agent = game.agent_selection
        agent_index = game.agents.index(agent)
        action = agents[agent_index].action()
        obs = game.observe(agent)
        print(f"Agent {agent} - Observation: {obs} - Action: {mapActionName(action) if n_games == 1 else None}")
        game.step(action)

    for agent in game.agents:
        cum_rewards[agent] += game.reward(agent)

print("Average rewards:")
for agent in game.agents:
    print(f"{agent}: {cum_rewards[agent] / n_games:.3f}")

  0%|          | 0/100 [00:00<?, ?it/s]

Agent player_0 - Observation: Q_#_1_2_0 - Action: None


  1%|          | 1/100 [00:03<06:22,  3.87s/it]

Agent player_1 - Observation: J_#_2_2_0c - Action: None
Agent player_0 - Observation: J_#_1_2_0 - Action: None


  3%|▎         | 3/100 [00:07<03:55,  2.43s/it]

...

Agent player_1 - Observation: K_#_6_4_0crr - Action: None
Agent player_0 - Observation: K_J_6_6_0crrc - Action: None


 98%|█████████▊| 98/100 [11:15<00:12,  6.16s/it]

Agent player_1 - Observation: J_#_2_1_1 - Action: None
Agent player_0 - Observation: Q_#_1_2_0 - Action: None


100%|██████████| 100/100 [11:18<00:00,  6.79s/it]

Agent player_1 - Observation: J_#_4_2_0r - Action: None
Agent player_0 - Observation: Q_#_1_2_0 - Action: None
Average rewards:
player_0: -1.525
player_1: 1.525
